# DeepGlobe Land Cover — Carregamento e Exploração

Exemplo de como **baixar, decodificar e visualizar** o dataset [DeepGlobe Land Cover Classification](https://www.kaggle.com/datasets/balraj98/deepglobe-land-cover-classification-dataset) para segmentação semântica de imagens de satélite (plantações, florestas, rios, etc.).

**7 classes** (codificadas por cor RGB nas máscaras):

| Classe | Cor (R,G,B) | Cobre |
|---|---|---|
| `urban_land` | (0,255,255) ciano | áreas urbanas |
| `agriculture_land` | (255,255,0) amarelo | lavouras, fazendas, pomares, vinhedos |
| `rangeland` | (255,0,255) magenta | pastagens / campo aberto |
| `forest_land` | (0,255,0) verde | florestas |
| `water` | (0,0,255) azul | rios, lagos, áreas alagadas |
| `barren_land` | (255,255,255) branco | solo exposto |
| `unknown` | (0,0,0) preto | nuvem / não rotulado |

> **Importante:** apenas o split `train` possui máscaras (`*_mask.png`). Os splits `valid` e `test` contêm somente as imagens de satélite (`*_sat.jpg`), pois eram usados para submissão no desafio original. Por isso usamos `train` para treino/validação.

## 1. Imports

Usa apenas bibliotecas já presentes no projeto: `kagglehub`, `opencv`, `numpy`, `matplotlib` e `torch`. Para treinar uma U-Net depois, veja a seção final (requer `segmentation-models-pytorch`).

In [ ]:
import csv
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline

## 2. Download do dataset

`kagglehub` faz o download e cacheia localmente (em `~/.cache/kagglehub`). Na primeira execução pode pedir autenticação do Kaggle (`~/.kaggle/kaggle.json`). São ~3.5 GB.

In [ ]:
import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("balraj98/deepglobe-land-cover-classification-dataset")
)
print("Dataset em:", dataset_path)
print("Conteúdo:", [p.name for p in dataset_path.iterdir()])

## 3. Mapa de classes (cor → índice)

Lemos o `class_dict.csv` que acompanha o dataset e montamos:
- `CLASS_NAMES`: lista de nomes na ordem do índice
- `CLASS_COLORS`: array `(n_classes, 3)` com a cor RGB de cada classe

In [ ]:
class_dict_path = dataset_path / "class_dict.csv"

CLASS_NAMES, colors = [], []
with open(class_dict_path) as f:
    for row in csv.DictReader(f):
        CLASS_NAMES.append(row["name"])
        colors.append([int(row["r"]), int(row["g"]), int(row["b"])])

CLASS_COLORS = np.array(colors, dtype=np.uint8)
N_CLASSES = len(CLASS_NAMES)

for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    print(f"{i}: {name:18s} RGB={tuple(color)}")

## 4. Listar os pares (imagem, máscara)

Cada amostra do split `train` tem `<id>_sat.jpg` (satélite) e `<id>_mask.png` (rótulo).

In [ ]:
# A pasta pode vir como 'train/' ou aninhada; procuramos as imagens *_sat.jpg recursivamente.
sat_images = sorted(dataset_path.rglob("*_sat.jpg"))

pairs = []
for sat in sat_images:
    mask = sat.with_name(sat.name.replace("_sat.jpg", "_mask.png"))
    if mask.exists():
        pairs.append((sat, mask))

print(f"Imagens de satélite encontradas: {len(sat_images)}")
print(f"Pares (imagem + máscara) válidos: {len(pairs)}")
print("Exemplo:", pairs[0][0].name, "->", pairs[0][1].name)

## 5. Funções de codificação/decodificação da máscara

As máscaras são RGB. Para treinar precisamos de um mapa de **índices de classe** (H×W com valores 0..6). E para visualizar fazemos o caminho inverso (índice → cor).

In [ ]:
def rgb_to_label(mask_rgb: np.ndarray) -> np.ndarray:
    """Converte máscara RGB (H,W,3) em mapa de índices de classe (H,W)."""
    label = np.zeros(mask_rgb.shape[:2], dtype=np.uint8)
    for idx, color in enumerate(CLASS_COLORS):
        matches = np.all(mask_rgb == color, axis=-1)
        label[matches] = idx
    return label


def label_to_rgb(label: np.ndarray) -> np.ndarray:
    """Converte mapa de índices (H,W) de volta para RGB (H,W,3) para visualização."""
    return CLASS_COLORS[label]


def load_sample(sat_path: Path, mask_path: Path):
    """Lê imagem (RGB) e máscara (índices) de um par."""
    image = cv2.cvtColor(cv2.imread(str(sat_path)), cv2.COLOR_BGR2RGB)
    mask_rgb = cv2.cvtColor(cv2.imread(str(mask_path)), cv2.COLOR_BGR2RGB)
    return image, rgb_to_label(mask_rgb)

## 6. Visualizar uma amostra

Mostra a imagem de satélite, a máscara colorida e a sobreposição (overlay).

In [ ]:
def show_sample(sat_path: Path, mask_path: Path, alpha: float = 0.45):
    image, label = load_sample(sat_path, mask_path)
    mask_rgb = label_to_rgb(label)
    overlay = cv2.addWeighted(image, 1 - alpha, mask_rgb, alpha, 0)

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    for ax, img, title in zip(
        axes, [image, mask_rgb, overlay], ["Satélite", "Máscara", "Overlay"]
    ):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    # Legenda apenas com as classes presentes nesta amostra
    present = np.unique(label)
    handles = [
        mpatches.Patch(color=CLASS_COLORS[i] / 255, label=CLASS_NAMES[i]) for i in present
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(present), bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.show()


show_sample(*pairs[0])

## 7. Distribuição das classes

Conta a proporção de pixels de cada classe numa amostra do dataset — útil para detectar desbalanceamento (no DeepGlobe, `agriculture` costuma dominar).

In [ ]:
N_SAMPLE = 50  # quantas imagens amostrar para estimar a distribuição

counts = np.zeros(N_CLASSES, dtype=np.int64)
for sat, mask in pairs[:N_SAMPLE]:
    _, label = load_sample(sat, mask)
    binc = np.bincount(label.ravel(), minlength=N_CLASSES)
    counts += binc

pct = 100 * counts / counts.sum()
order = np.argsort(pct)[::-1]

plt.figure(figsize=(9, 4))
plt.bar(
    [CLASS_NAMES[i] for i in order],
    pct[order],
    color=[CLASS_COLORS[i] / 255 for i in order],
    edgecolor="black",
)
plt.ylabel("% de pixels")
plt.title(f"Distribuição de classes ({N_SAMPLE} imagens)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

for i in order:
    print(f"{CLASS_NAMES[i]:18s} {pct[i]:5.1f}%")

## 8. `Dataset` PyTorch

Encapsula o carregamento num `torch.utils.data.Dataset` pronto para treino: redimensiona, normaliza a imagem para `[0,1]` no formato `CHW` e devolve a máscara como índices `long`. As imagens originais têm 2448×2448 px — redimensionamos para algo treinável (ex.: 512×512).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class DeepGlobeDataset(Dataset):
    def __init__(self, pairs, size=512):
        self.pairs = pairs
        self.size = size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sat_path, mask_path = self.pairs[idx]
        image, label = load_sample(sat_path, mask_path)

        # INTER_AREA p/ a imagem (suave), INTER_NEAREST p/ a máscara (preserva os índices)
        image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_AREA)
        label = cv2.resize(label, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0  # (3,H,W)
        label = torch.from_numpy(label).long()                            # (H,W)
        return image, label


# Split simples treino/validação (80/20)
split = int(0.8 * len(pairs))
train_ds = DeepGlobeDataset(pairs[:split])
val_ds = DeepGlobeDataset(pairs[split:])
print(f"Treino: {len(train_ds)}  |  Validação: {len(val_ds)}")

# Teste rápido: shape de um batch
loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
images, labels = next(iter(loader))
print("Batch imagens:", images.shape, images.dtype, "| min/max:", float(images.min()), float(images.max()))
print("Batch máscaras:", labels.shape, labels.dtype, "| classes no batch:", torch.unique(labels).tolist())

## 9. Treinar uma U-Net

O `Dataset` da seção 8 já alimenta o modelo diretamente. Usamos a [`segmentation-models-pytorch`](https://github.com/qubvel-org/segmentation_models.pytorch) (`smp`), já adicionada ao projeto, para montar uma **U-Net com encoder ResNet-34 pré-treinado na ImageNet**.

> Treino real consome bastante memória. Ajuste `BATCH_SIZE`, `size` (no `DeepGlobeDataset`) e `EPOCHS` conforme sua GPU/CPU. Sem GPU, reduza para poucas imagens só para validar o fluxo.

### 9.1 Modelo, loss e otimizador

In [ ]:
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

BATCH_SIZE = 4
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",  # baixa os pesos do encoder na 1ª vez
    in_channels=3,
    classes=N_CLASSES,
).to(device)

# Dice lida bem com o desbalanceamento; somamos CrossEntropy para estabilizar o treino.
dice_loss = smp.losses.DiceLoss(mode="multiclass")
ce_loss = torch.nn.CrossEntropyLoss()


def criterion(logits, target):
    return dice_loss(logits, target) + ce_loss(logits, target)


optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parâmetros")

### 9.2 Métrica — mIoU

A métrica padrão para segmentação é o **mIoU** (mean Intersection over Union). Calculamos o IoU por classe e tiramos a média, ignorando classes que não aparecem no conjunto avaliado.

In [ ]:
@torch.no_grad()
def evaluate_miou(model, loader, n_classes=N_CLASSES):
    """mIoU acumulando interseção/união por classe em todo o loader.

    Classes ausentes do conjunto (união 0) são ignoradas na média.
    """
    model.eval()
    inter = torch.zeros(n_classes, device=device)
    union = torch.zeros(n_classes, device=device)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)  # (B, H, W)
        for c in range(n_classes):
            p, t = preds == c, labels == c
            inter[c] += (p & t).sum()
            union[c] += (p | t).sum()
    iou = inter / union.clamp(min=1)
    miou = iou[union > 0].mean().item()
    return miou, iou.cpu()

### 9.3 Loop de treino

Treina por algumas épocas, calcula o `mIoU` de validação ao final de cada uma e salva o melhor modelo em `unet_deepglobe.pt`.

In [ ]:
EPOCHS = 5  # aumente para resultados melhores

best_miou = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)            # (B, N_CLASSES, H, W)
        loss = criterion(logits, labels)  # labels: (B, H, W) long
        loss.backward()
        optimizer.step()
        running += loss.item()

    train_loss = running / len(train_loader)
    miou, iou = evaluate_miou(model, val_loader)
    flag = ""
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), "unet_deepglobe.pt")
        flag = "  <- salvo (melhor)"
    print(f"epoch {epoch}/{EPOCHS}  loss={train_loss:.4f}  val_mIoU={miou:.4f}{flag}")

print("\nIoU por classe (última época):")
for name, v in zip(CLASS_NAMES, iou):
    print(f"  {name:18s} {v:.3f}")

### 9.4 Visualizar predições

Compara, lado a lado, a imagem de satélite, a máscara real (ground truth) e a predição do modelo numa amostra de validação.

In [ ]:
@torch.no_grad()
def show_prediction(idx, dataset=val_ds, alpha=0.45):
    """Compara satélite, ground truth e predição do modelo para uma amostra."""
    model.eval()
    image, label = dataset[idx]
    logits = model(image.unsqueeze(0).to(device))
    pred = logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

    img_np = (image.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    gt_rgb = label_to_rgb(label.numpy())
    pred_rgb = label_to_rgb(pred)

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))
    for ax, img, title in zip(
        axes, [img_np, gt_rgb, pred_rgb], ["Satélite", "Ground truth", "Predição"]
    ):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_prediction(0)

## 10. Como melhorar os resultados

- **Data augmentation** (flips, rotações, brilho) com `albumentations` — aplicar a *mesma* transformação à imagem e à máscara.
- **Patches**: em vez de redimensionar 2448→512 (perde detalhe), recortar tiles de 512×512 da imagem original.
- **Desbalanceamento**: a `DiceLoss` já ajuda; também dá para ponderar a `CrossEntropyLoss` pelas frequências da seção 7.
- **Normalização do encoder**: para extrair o máximo do ResNet pré-treinado, normalize com a média/desvio da ImageNet (`smp.encoders.get_preprocessing_fn`).
- **Scheduler + mais épocas**: `CosineAnnealingLR` e 30–50 épocas costumam elevar bastante o mIoU.
- **Salvar o melhor modelo**: `torch.save(model.state_dict(), "unet_deepglobe.pt")` quando o `val_mIoU` melhorar.